# Boltzmann Populations and Conformational Analysis

This notebook covers the statistical mechanics of conformer distributions, essential for:

- **NMR analysis** - Boltzmann-averaged coupling constants and NOEs
- **Drug binding** - accessible conformations for receptor interaction
- **Property prediction** - ensemble-averaged molecular properties
- **Free energy calculations** - conformational entropy contributions

## Statistical Mechanics Background

### Boltzmann Distribution

At thermal equilibrium, the population of conformer $i$ is:

$$p_i = \frac{e^{-E_i/k_BT}}{\sum_j e^{-E_j/k_BT}} = \frac{e^{-E_i/k_BT}}{Z}$$

where $Z$ is the partition function.

### Useful Numbers at 298K

| ΔE (kcal/mol) | Population Ratio |
|---------------|------------------|
| 0.0 | 1.0 : 1.0 |
| 0.6 | 2.7 : 1.0 |
| 1.0 | 5.4 : 1.0 |
| 1.36 | 10 : 1 |
| 2.0 | 29 : 1 |
| 2.72 | 100 : 1 |

Rule of thumb: **1.36 kcal/mol = 10:1 ratio at room temperature**

In [ ]:
import os
import tempfile
from pathlib import Path

import numpy as np
import Auto3D
from Auto3D import Auto3DOptions, main
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem import TorsionFingerprints
import pandas as pd

print(f"Auto3D version: {Auto3D.__version__}")

# Constants
R = 1.987e-3  # kcal/(mol·K)
T = 298.15    # K
RT = R * T    # ~0.593 kcal/mol at 298K

## 1. Basic Boltzmann Analysis

Let's analyze conformer populations for a flexible drug molecule.

In [ ]:
def calculate_boltzmann_populations(energies_kcal, T=298.15):
    """
    Calculate Boltzmann populations from conformer energies.
    
    Args:
        energies_kcal: Array of conformer energies in kcal/mol
        T: Temperature in Kelvin
        
    Returns:
        Array of populations (sum to 1.0)
    """
    energies = np.array(energies_kcal)
    RT = R * T
    
    # Shift to prevent numerical overflow
    energies_shifted = energies - energies.min()
    
    # Boltzmann weights
    weights = np.exp(-energies_shifted / RT)
    
    # Normalize to populations
    populations = weights / weights.sum()
    
    return populations


def calculate_conformational_entropy(populations):
    """
    Calculate conformational entropy from populations.
    
    S_conf = -R * Σ p_i * ln(p_i)
    
    Returns entropy in cal/(mol·K)
    """
    # Filter out zero populations
    p = populations[populations > 0]
    S_conf = -R * 1000 * np.sum(p * np.log(p))  # Convert to cal/(mol·K)
    return S_conf


# Example: hypothetical conformer energies
example_energies = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0]

pops = calculate_boltzmann_populations(example_energies)
S_conf = calculate_conformational_entropy(pops)

print("Boltzmann Population Analysis:")
print("-" * 50)
for i, (e, p) in enumerate(zip(example_energies, pops)):
    bar = "█" * int(p * 50)
    print(f"Conf {i+1}: E={e:4.1f} kcal/mol, pop={p*100:5.1f}% {bar}")
print("-" * 50)
print(f"Conformational entropy: {S_conf:.2f} cal/(mol·K)")
print(f"-T*S_conf at 298K: {-T * S_conf / 1000:.2f} kcal/mol")

## 2. Conformational Analysis of a Drug Molecule

Let's analyze a real drug molecule with Auto3D.

In [ ]:
# Metoclopramide - a flexible drug with multiple rotatable bonds
# Used as antiemetic
drug_smiles = {
    "metoclopramide": "CCN(CC)CCNC(=O)C1=CC(=C(C=C1OC)N)Cl",
}

# Count rotatable bonds
mol = Chem.MolFromSmiles(drug_smiles["metoclopramide"])
n_rot = Descriptors.NumRotatableBonds(mol)

print(f"Drug: Metoclopramide")
print(f"Rotatable bonds: {n_rot}")
print(f"Theoretical max conformers: ~{3**n_rot} (3 states per bond)")

In [ ]:
# Write to file
with tempfile.NamedTemporaryFile(mode='w', suffix='.smi', delete=False) as f:
    for name, smi in drug_smiles.items():
        f.write(f"{smi} {name}\n")
    drug_file = f.name

print(f"Input file: {drug_file}")

In [ ]:
# Generate multiple conformers with Auto3D
# Use window mode to get all conformers within energy range

if __name__ == "__main__":
    config = Auto3DOptions(
        path=drug_file,
        window=5.0,              # All conformers within 5 kcal/mol
        optimizing_engine="AIMNET",
        threshold=0.5,           # RMSD threshold for diversity
        max_confs=300,           # Generate enough initial conformers
        use_gpu=True,
    )
    
    conformer_output = main(config)
    print(f"Output: {conformer_output}")

In [ ]:
def analyze_conformer_ensemble(sdf_path):
    """
    Comprehensive analysis of conformer ensemble.
    """
    mols = list(Chem.SDMolSupplier(sdf_path, removeHs=False))
    
    # Extract energies
    energies = []
    for mol in mols:
        if mol is None:
            continue
        if mol.HasProp("E_rel"):
            e = float(mol.GetProp("E_rel"))
        elif mol.HasProp("E_hartree"):
            e = float(mol.GetProp("E_hartree")) * 627.509
        else:
            continue
        energies.append(e)
    
    energies = np.array(energies)
    
    # Make relative
    if energies.min() > 0:  # Already relative
        energies_rel = energies
    else:
        energies_rel = energies - energies.min()
    
    # Calculate populations
    populations = calculate_boltzmann_populations(energies_rel)
    
    # Statistics
    S_conf = calculate_conformational_entropy(populations)
    
    # Effective number of conformers
    # (Perplexity - how many conformers are "effectively" populated)
    effective_n = np.exp(-np.sum(populations * np.log(populations + 1e-10)))
    
    return {
        "n_conformers": len(energies),
        "energies_rel": energies_rel,
        "populations": populations,
        "S_conf": S_conf,
        "effective_n": effective_n,
        "e_range": energies_rel.max() - energies_rel.min(),
        "mols": mols
    }


if 'conformer_output' in dir():
    analysis = analyze_conformer_ensemble(conformer_output)
    
    print(f"Conformer Ensemble Analysis:")
    print("=" * 50)
    print(f"Total conformers: {analysis['n_conformers']}")
    print(f"Energy range: {analysis['e_range']:.2f} kcal/mol")
    print(f"Conformational entropy: {analysis['S_conf']:.2f} cal/(mol·K)")
    print(f"Effective conformers: {analysis['effective_n']:.1f}")
    print("=" * 50)

In [ ]:
# Show top conformers
if 'analysis' in dir():
    print("\nTop 10 Conformers by Population:")
    print("-" * 50)
    
    # Sort by population
    sorted_idx = np.argsort(analysis['populations'])[::-1]
    
    cumulative = 0
    for rank, idx in enumerate(sorted_idx[:10]):
        e = analysis['energies_rel'][idx]
        p = analysis['populations'][idx]
        cumulative += p
        bar = "█" * int(p * 50)
        print(f"#{rank+1:2d}: E={e:5.2f} kcal/mol, pop={p*100:5.1f}%, cum={cumulative*100:5.1f}% {bar}")
    
    print("-" * 50)
    print(f"Top 10 conformers account for {cumulative*100:.1f}% of population")

## 3. Boltzmann-Weighted Properties

For comparison with experimental observables, properties should be Boltzmann-averaged.

In [ ]:
def boltzmann_average_property(values, populations):
    """
    Calculate Boltzmann-weighted average of a property.
    
    <P> = Σ p_i * P_i
    """
    return np.sum(values * populations)


def calculate_conformer_properties(mols, populations):
    """
    Calculate Boltzmann-averaged molecular properties.
    """
    # Calculate properties for each conformer
    dipoles = []  # Would need QM for accurate dipoles
    radii_gyration = []
    asphericity = []
    
    for mol in mols:
        if mol is None:
            continue
            
        # Radius of gyration
        try:
            rg = rdMolDescriptors.CalcRadiusOfGyration(mol)
            radii_gyration.append(rg)
        except:
            radii_gyration.append(np.nan)
        
        # Asphericity (0 = sphere, 1 = rod)
        try:
            asp = rdMolDescriptors.CalcAsphericity(mol)
            asphericity.append(asp)
        except:
            asphericity.append(np.nan)
    
    radii_gyration = np.array(radii_gyration)
    asphericity = np.array(asphericity)
    
    # Filter NaN
    valid = ~np.isnan(radii_gyration) & ~np.isnan(asphericity)
    
    if valid.sum() > 0:
        rg_avg = boltzmann_average_property(radii_gyration[valid], populations[valid])
        asp_avg = boltzmann_average_property(asphericity[valid], populations[valid])
        
        return {
            "R_gyration_avg": rg_avg,
            "asphericity_avg": asp_avg,
            "R_gyration_range": (radii_gyration[valid].min(), radii_gyration[valid].max()),
            "asphericity_range": (asphericity[valid].min(), asphericity[valid].max()),
        }
    
    return None


if 'analysis' in dir():
    props = calculate_conformer_properties(
        analysis['mols'], 
        analysis['populations']
    )
    
    if props:
        print("\nBoltzmann-Averaged Properties:")
        print(f"  Radius of gyration: {props['R_gyration_avg']:.2f} Å")
        print(f"    Range: {props['R_gyration_range'][0]:.2f} - {props['R_gyration_range'][1]:.2f} Å")
        print(f"  Asphericity: {props['asphericity_avg']:.3f}")
        print(f"    Range: {props['asphericity_range'][0]:.3f} - {props['asphericity_range'][1]:.3f}")

## 4. Temperature Effects on Populations

Higher temperatures flatten the population distribution.

In [ ]:
def analyze_temperature_dependence(energies_rel, temperatures):
    """
    Analyze how conformer populations change with temperature.
    """
    results = []
    
    for T in temperatures:
        pops = calculate_boltzmann_populations(energies_rel, T)
        S_conf = calculate_conformational_entropy(pops)
        effective_n = np.exp(-np.sum(pops * np.log(pops + 1e-10)))
        
        # Population of global minimum
        p_global_min = pops[np.argmin(energies_rel)]
        
        results.append({
            "T": T,
            "S_conf": S_conf,
            "effective_n": effective_n,
            "p_global_min": p_global_min * 100
        })
    
    return pd.DataFrame(results)


if 'analysis' in dir():
    temps = [200, 250, 298, 310, 350, 400, 500]
    df_temp = analyze_temperature_dependence(analysis['energies_rel'], temps)
    
    print("\nTemperature Dependence of Conformer Distribution:")
    print("-" * 60)
    print(df_temp.to_string(index=False, float_format="%.1f"))
    print("-" * 60)
    print("\nNote: At higher T, populations become more uniform.")
    print("Body temperature (310K) vs room temperature (298K) matters!")

## 5. Conformational Clustering

Group similar conformers to identify distinct conformational families.

In [ ]:
def cluster_conformers_by_rmsd(mols, populations, rmsd_threshold=1.0):
    """
    Cluster conformers by RMSD similarity.
    
    Returns list of clusters, each with representative and total population.
    """
    from rdkit.Chem import AllChem
    
    n_mols = len(mols)
    
    # Simple greedy clustering
    clusters = []  # [(representative_idx, [member_indices])]
    assigned = set()
    
    # Sort by population (start with most populated)
    sorted_idx = np.argsort(populations)[::-1]
    
    for i in sorted_idx:
        if i in assigned:
            continue
        if mols[i] is None:
            continue
            
        # Start new cluster
        cluster_members = [i]
        assigned.add(i)
        
        # Find similar conformers
        for j in sorted_idx:
            if j in assigned or mols[j] is None:
                continue
            
            try:
                rmsd = AllChem.GetBestRMS(mols[i], mols[j])
                if rmsd < rmsd_threshold:
                    cluster_members.append(j)
                    assigned.add(j)
            except:
                continue
        
        clusters.append((i, cluster_members))
    
    # Calculate cluster populations
    cluster_data = []
    for rep_idx, members in clusters:
        total_pop = sum(populations[m] for m in members)
        cluster_data.append({
            "representative": rep_idx,
            "n_members": len(members),
            "population": total_pop * 100
        })
    
    return pd.DataFrame(cluster_data).sort_values("population", ascending=False)


if 'analysis' in dir() and len(analysis['mols']) > 1:
    try:
        df_clusters = cluster_conformers_by_rmsd(
            analysis['mols'], 
            analysis['populations'],
            rmsd_threshold=1.0
        )
        
        print("\nConformational Families (RMSD < 1.0 Å):")
        print("-" * 50)
        print(df_clusters.head(10).to_string(index=False))
        print("-" * 50)
        print(f"Total families: {len(df_clusters)}")
    except Exception as e:
        print(f"Clustering failed: {e}")

## 6. Free Energy Surface

The Boltzmann-weighted free energy is:

$$G = -RT\ln Z = -RT\ln\sum_i e^{-E_i/RT}$$

This is the reference for calculating free energy differences.

In [ ]:
def calculate_free_energy_from_ensemble(energies_rel, T=298.15):
    """
    Calculate the configurational free energy from conformer ensemble.
    
    G = -RT ln(Z) where Z = Σ exp(-E_i/RT)
    """
    RT = R * T
    
    # Partition function
    Z = np.sum(np.exp(-energies_rel / RT))
    
    # Free energy (relative to global minimum)
    G = -RT * np.log(Z)
    
    # Entropy contribution
    populations = calculate_boltzmann_populations(energies_rel, T)
    S_conf = calculate_conformational_entropy(populations) / 1000  # kcal/(mol·K)
    
    # Average energy
    E_avg = np.sum(populations * energies_rel)
    
    return {
        "G": G,
        "E_avg": E_avg,
        "S_conf": S_conf * 1000,  # cal/(mol·K)
        "-T*S": -T * S_conf,
        "Z": Z
    }


if 'analysis' in dir():
    fe = calculate_free_energy_from_ensemble(analysis['energies_rel'])
    
    print("\nFree Energy Analysis:")
    print("=" * 50)
    print(f"Average energy <E>:         {fe['E_avg']:>8.2f} kcal/mol")
    print(f"Conformational entropy S:   {fe['S_conf']:>8.2f} cal/(mol·K)")
    print(f"Entropy contribution -TS:   {fe['-T*S']:>8.2f} kcal/mol")
    print(f"Free energy G (rel):        {fe['G']:>8.2f} kcal/mol")
    print(f"Partition function Z:       {fe['Z']:>8.2f}")
    print("=" * 50)
    print("\nNote: G = <E> - TS (approximately)")

## 7. Comparison with Single-Conformer Analysis

Demonstrates the importance of ensemble averaging.

In [ ]:
if 'analysis' in dir():
    # Compare single conformer vs ensemble
    props = calculate_conformer_properties(
        analysis['mols'], 
        analysis['populations']
    )
    
    if props and len(analysis['mols']) > 0:
        # Single conformer (global minimum)
        idx_min = np.argmin(analysis['energies_rel'])
        mol_min = analysis['mols'][idx_min]
        
        if mol_min:
            try:
                rg_single = rdMolDescriptors.CalcRadiusOfGyration(mol_min)
                asp_single = rdMolDescriptors.CalcAsphericity(mol_min)
                
                print("\nSingle Conformer vs Ensemble Comparison:")
                print("-" * 50)
                print(f"{'Property':<25} {'Single':<12} {'Ensemble':<12}")
                print("-" * 50)
                print(f"{'Radius of gyration (Å)':<25} {rg_single:<12.2f} {props['R_gyration_avg']:<12.2f}")
                print(f"{'Asphericity':<25} {asp_single:<12.3f} {props['asphericity_avg']:<12.3f}")
                print("-" * 50)
                print("\nUsing only the global minimum can give misleading results!")
            except Exception as e:
                print(f"Property calculation failed: {e}")

## 8. Practical Applications

### NMR Analysis
- J-couplings depend on dihedral angles
- NOE intensities depend on interatomic distances
- Observed values are Boltzmann averages

### Drug Design
- Binding affinities should include conformational penalty
- Rigid molecules have entropic advantage
- Bioactive conformation may not be global minimum

### QSAR/ML
- Use Boltzmann-averaged descriptors
- Consider conformational entropy as descriptor
- Multiple conformers may be needed for 3D-QSAR

In [ ]:
# Cleanup
if 'drug_file' in dir() and os.path.exists(drug_file):
    os.unlink(drug_file)

## Summary

This tutorial covered:

1. **Boltzmann distribution** - population = exp(-E/RT) / Z
2. **Conformational entropy** - S = -R Σ p·ln(p)
3. **Ensemble analysis** - effective number of conformers
4. **Temperature effects** - higher T = flatter distribution
5. **Boltzmann averaging** - <property> = Σ p·property
6. **Free energy** - G = -RT ln(Z)

Key takeaways:
- **1.36 kcal/mol ≈ 10:1 ratio** at room temperature
- **Ensemble matters** - single conformer can be misleading
- **Temperature matters** - body temp (310K) vs room temp (298K)